# Bureau Feature Engineering

This notebook builds applicant-level features from the cleaned bureau and bureau-balance data. It first summarizes the monthly bureau-balance history to one row per credit account, merges that into the bureau table, then aggregates everything up to one row per applicant.


## Import libraries


In [3]:
from pathlib import Path

import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)
print("Libraries imported successfully.")

Libraries imported successfully.


## Set project paths


In [5]:
current_folder = Path.cwd().resolve()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
bureau_path = project_root / "data" / "interim" / "bureau_clean.pkl"
balance_path = project_root / "data" / "interim" / "bureau_balance_clean.pkl"
application_path = project_root / "data" / "interim" / "application_clean.pkl"
training_ids_path = project_root / "data" / "modeling" / "splits" / "training_ids.csv"
test_ids_path = project_root / "data" / "modeling" / "splits" / "test_ids.csv"
output_path = project_root / "data" / "features" / "bureau_features.pkl"
audit_folder = project_root / "reports" / "audits"
output_path.parent.mkdir(parents=True, exist_ok=True)
audit_folder.mkdir(parents=True, exist_ok=True)
for required_path in [bureau_path, balance_path, application_path, training_ids_path, test_ids_path]:
    assert required_path.exists(), f"Required file was not found: {required_path}"
print("Bureau input:", bureau_path)
print("Bureau-balance input:", balance_path)
print("Feature output:", output_path)

Bureau input: /Users/taranveersingh/A-MRP/data/interim/bureau_clean.pkl
Bureau-balance input: /Users/taranveersingh/A-MRP/data/interim/bureau_balance_clean.pkl
Feature output: /Users/taranveersingh/A-MRP/data/features/bureau_features.pkl


## Load cleaned data and split information


In [7]:
bureau = pd.read_pickle(bureau_path)
balance = pd.read_pickle(balance_path)
application_target = pd.read_pickle(application_path)[["SK_ID_CURR", "TARGET"]]
training_ids = pd.read_csv(training_ids_path)["SK_ID_CURR"]
test_ids = pd.read_csv(test_ids_path)["SK_ID_CURR"]
training_id_set = set(training_ids)
test_id_set = set(test_ids)
print("Clean bureau shape:", bureau.shape)
print("Clean bureau-balance shape:", balance.shape)
print("Applicants with bureau records:", bureau["SK_ID_CURR"].nunique())
print("Applicants with monthly bureau history:", balance["SK_ID_CURR"].nunique())

Clean bureau shape: (1465325, 22)
Clean bureau-balance shape: (14701612, 10)
Applicants with bureau records: 263491
Applicants with monthly bureau history: 92231


Only about a third of applicants with bureau records (92,231 of 263,491) actually have monthly bureau-balance history, the rest are credits with no monthly record at all.


## Aggregate monthly history to one row per bureau account


In [10]:
balance_account = balance.groupby("SK_ID_BUREAU").agg(
    BB_MONTH_COUNT=("MONTHS_BALANCE", "count"),
    BB_OLDEST_MONTH=("MONTHS_BALANCE", "min"),
    BB_LATEST_MONTH=("MONTHS_BALANCE", "max"),
    BB_DELINQUENT_MONTHS=("BB_STATUS_DELINQUENT", "sum"),
    BB_DELINQUENT_RATE=("BB_STATUS_DELINQUENT", "mean"),
    BB_CLOSED_RATE=("BB_STATUS_CLOSED", "mean"),
    BB_UNKNOWN_RATE=("BB_STATUS_UNKNOWN", "mean"),
    BB_MAX_DELINQUENCY_LEVEL=("BB_DELINQUENCY_LEVEL", "max"),
    BB_MEAN_DELINQUENCY_LEVEL=("BB_DELINQUENCY_LEVEL", "mean"),
    BB_RECORD_MISSING_RATE_MEAN=("BB_RECORD_MISSING_RATE", "mean"),
).reset_index()

recent_balance = balance.loc[balance["MONTHS_BALANCE"].ge(-12)]
recent_account = recent_balance.groupby("SK_ID_BUREAU").agg(
    BB_RECENT_12M_COUNT=("MONTHS_BALANCE", "count"),
    BB_RECENT_12M_DELINQUENT_MONTHS=("BB_STATUS_DELINQUENT", "sum"),
    BB_RECENT_12M_DELINQUENT_RATE=("BB_STATUS_DELINQUENT", "mean"),
).reset_index()
balance_account = balance_account.merge(recent_account, on="SK_ID_BUREAU", how="left", validate="one_to_one")
for column in ["BB_RECENT_12M_COUNT", "BB_RECENT_12M_DELINQUENT_MONTHS", "BB_RECENT_12M_DELINQUENT_RATE"]:
    balance_account[column] = balance_account[column].fillna(0)
balance_account["BB_SEVERITY_AVAILABLE"] = balance_account["BB_MAX_DELINQUENCY_LEVEL"].notna().astype("int8")
balance_account[["BB_MAX_DELINQUENCY_LEVEL", "BB_MEAN_DELINQUENCY_LEVEL"]] = balance_account[["BB_MAX_DELINQUENCY_LEVEL", "BB_MEAN_DELINQUENCY_LEVEL"]].fillna(0)
del balance, recent_balance, recent_account
print("Bureau accounts summarized:", len(balance_account))
print("Account summary columns:", balance_account.shape[1])

Bureau accounts summarized: 523515
Account summary columns: 15


This turns the monthly bureau-balance records into one row per credit account, describing things like how many months of history exist and how often the account was delinquent.


## Merge monthly summaries with cleaned bureau accounts


In [13]:
bureau_enriched = bureau.merge(balance_account, on="SK_ID_BUREAU", how="left", validate="one_to_one")
bureau_enriched["BUREAU_HAS_BALANCE_HISTORY"] = bureau_enriched["BB_MONTH_COUNT"].notna().astype("int8")
balance_feature_columns = [c for c in balance_account.columns if c != "SK_ID_BUREAU"]
bureau_enriched[balance_feature_columns] = bureau_enriched[balance_feature_columns].fillna(0)
print("Enriched bureau shape:", bureau_enriched.shape)
print("Accounts with monthly history:", int(bureau_enriched["BUREAU_HAS_BALANCE_HISTORY"].sum()))

Enriched bureau shape: (1465325, 37)
Accounts with monthly history: 523515


Accounts with no monthly bureau-balance history get their balance-related features filled with 0, since there is genuinely nothing to summarize for them. A flag column keeps track of which accounts actually had monthly history.


## Create bureau-account features


In [16]:
def safe_ratio(numerator, denominator):
    return (numerator / denominator.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)

bureau_enriched["BUREAU_DEBT_CREDIT_RATIO"] = safe_ratio(bureau_enriched["AMT_CREDIT_SUM_DEBT"], bureau_enriched["AMT_CREDIT_SUM"])
bureau_enriched["BUREAU_OVERDUE_CREDIT_RATIO"] = safe_ratio(bureau_enriched["AMT_CREDIT_SUM_OVERDUE"], bureau_enriched["AMT_CREDIT_SUM"])
bureau_enriched["BUREAU_CREDIT_AGE_YEARS"] = -bureau_enriched["DAYS_CREDIT"] / 365.25
bureau_enriched["BUREAU_REMAINING_DAYS"] = bureau_enriched["DAYS_CREDIT_ENDDATE"]
bureau_enriched["BUREAU_IS_ACTIVE"] = bureau_enriched["CREDIT_ACTIVE"].eq("Active").astype("int8")
bureau_enriched["BUREAU_IS_CLOSED"] = bureau_enriched["CREDIT_ACTIVE"].eq("Closed").astype("int8")
bureau_enriched["BUREAU_HAS_OVERDUE"] = (
    bureau_enriched["CREDIT_DAY_OVERDUE"].gt(0)
    | bureau_enriched["AMT_CREDIT_SUM_OVERDUE"].gt(0)
).astype("int8")
bureau_enriched["BUREAU_RECENT_CREDIT"] = bureau_enriched["DAYS_CREDIT"].ge(-365).astype("int8")
print("Account-level features created: 8")

Account-level features created: 8


These describe things like debt relative to credit amount, how old the credit is, and whether it is currently active, closed, or overdue.


## Aggregate numerical behaviour to applicant level


In [19]:
bureau_features = bureau_enriched.groupby("SK_ID_CURR").agg(
    BUREAU_ACCOUNT_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_ACTIVE_COUNT=("BUREAU_IS_ACTIVE", "sum"),
    BUREAU_ACTIVE_RATE=("BUREAU_IS_ACTIVE", "mean"),
    BUREAU_CLOSED_COUNT=("BUREAU_IS_CLOSED", "sum"),
    BUREAU_CLOSED_RATE=("BUREAU_IS_CLOSED", "mean"),
    BUREAU_OVERDUE_ACCOUNT_COUNT=("BUREAU_HAS_OVERDUE", "sum"),
    BUREAU_OVERDUE_ACCOUNT_RATE=("BUREAU_HAS_OVERDUE", "mean"),
    BUREAU_RECENT_CREDIT_COUNT=("BUREAU_RECENT_CREDIT", "sum"),
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
    BUREAU_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
    BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
    BUREAU_CREDIT_AGE_YEARS_MEAN=("BUREAU_CREDIT_AGE_YEARS", "mean"),
    BUREAU_CREDIT_SUM_TOTAL=("AMT_CREDIT_SUM", "sum"),
    BUREAU_CREDIT_SUM_MEAN=("AMT_CREDIT_SUM", "mean"),
    BUREAU_CREDIT_SUM_MAX=("AMT_CREDIT_SUM", "max"),
    BUREAU_DEBT_TOTAL=("AMT_CREDIT_SUM_DEBT", "sum"),
    BUREAU_DEBT_MEAN=("AMT_CREDIT_SUM_DEBT", "mean"),
    BUREAU_DEBT_MAX=("AMT_CREDIT_SUM_DEBT", "max"),
    BUREAU_LIMIT_TOTAL=("AMT_CREDIT_SUM_LIMIT", "sum"),
    BUREAU_OVERDUE_TOTAL=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_OVERDUE_MAX=("AMT_CREDIT_SUM_OVERDUE", "max"),
    BUREAU_DAYS_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
    BUREAU_PROLONG_TOTAL=("CNT_CREDIT_PROLONG", "sum"),
    BUREAU_DEBT_CREDIT_RATIO_MEAN=("BUREAU_DEBT_CREDIT_RATIO", "mean"),
    BUREAU_DEBT_CREDIT_RATIO_MAX=("BUREAU_DEBT_CREDIT_RATIO", "max"),
    BUREAU_OVERDUE_CREDIT_RATIO_MAX=("BUREAU_OVERDUE_CREDIT_RATIO", "max"),
    BUREAU_RECORD_MISSING_RATE_MEAN=("BUREAU_RECORD_MISSING_RATE", "mean"),
    BUREAU_DEBT_ABOVE_CREDIT_COUNT=("BUREAU_DEBT_ABOVE_CREDIT", "sum"),
    BUREAU_OVERDUE_INCONSISTENCY_COUNT=("BUREAU_OVERDUE_INCONSISTENCY", "sum"),
    BUREAU_BALANCE_HISTORY_ACCOUNT_COUNT=("BUREAU_HAS_BALANCE_HISTORY", "sum"),
    BUREAU_BALANCE_HISTORY_ACCOUNT_RATE=("BUREAU_HAS_BALANCE_HISTORY", "mean"),
    BUREAU_MONTH_RECORD_COUNT=("BB_MONTH_COUNT", "sum"),
    BUREAU_DELINQUENT_MONTH_COUNT=("BB_DELINQUENT_MONTHS", "sum"),
    BUREAU_DELINQUENT_RATE_MEAN=("BB_DELINQUENT_RATE", "mean"),
    BUREAU_MAX_DELINQUENCY_LEVEL=("BB_MAX_DELINQUENCY_LEVEL", "max"),
    BUREAU_UNKNOWN_STATUS_RATE_MEAN=("BB_UNKNOWN_RATE", "mean"),
    BUREAU_RECENT_12M_DELINQUENT_COUNT=("BB_RECENT_12M_DELINQUENT_MONTHS", "sum"),
    BUREAU_RECENT_12M_DELINQUENT_RATE_MAX=("BB_RECENT_12M_DELINQUENT_RATE", "max"),
).reset_index()
bureau_features["BUREAU_TOTAL_DEBT_CREDIT_RATIO"] = safe_ratio(bureau_features["BUREAU_DEBT_TOTAL"], bureau_features["BUREAU_CREDIT_SUM_TOTAL"])
bureau_features["BUREAU_TOTAL_OVERDUE_DEBT_RATIO"] = safe_ratio(bureau_features["BUREAU_OVERDUE_TOTAL"], bureau_features["BUREAU_DEBT_TOTAL"])
print("Applicant numerical feature shape:", bureau_features.shape)

Applicant numerical feature shape: (263491, 41)


Each applicant's bureau accounts are summarized together here, things like total accounts, how many are active or overdue, and average debt and credit amounts.


## Add credit-type proportions


In [22]:
def clean_feature_name(value):
    return re.sub(r"[^A-Z0-9]+", "_", str(value).upper()).strip("_")

credit_type_table = pd.crosstab(
    bureau_enriched["SK_ID_CURR"], bureau_enriched["CREDIT_TYPE"], normalize="index"
)
credit_type_table.columns = ["BUREAU_CREDIT_TYPE_RATE_" + clean_feature_name(c) for c in credit_type_table.columns]
credit_type_table = credit_type_table.reset_index()
bureau_features = bureau_features.merge(credit_type_table, on="SK_ID_CURR", how="left", validate="one_to_one")
print("Credit-type proportion features added:", credit_type_table.shape[1] - 1)
print("Applicant feature shape:", bureau_features.shape)

Credit-type proportion features added: 10
Applicant feature shape: (263491, 51)


This adds the share of each applicant's bureau credits that fall into each credit type (like consumer credit or credit card), rather than just the raw counts.


## Apply training-only missingness and constant-feature rules


In [25]:
MISSING_THRESHOLD = 0.50
training_base = pd.DataFrame({"SK_ID_CURR": training_ids}).merge(
    application_target, on="SK_ID_CURR", how="left", validate="one_to_one"
).merge(bureau_features, on="SK_ID_CURR", how="left", validate="one_to_one")
decision_rows = []
for feature in [c for c in bureau_features.columns if c != "SK_ID_CURR"]:
    series = training_base[feature]
    missing_rate = series.isna().mean()
    unique_non_missing = series.nunique(dropna=True)
    correlation = series.corr(training_base["TARGET"]) if unique_non_missing > 1 else np.nan
    decision = "Keep"
    reason = "Retain for global cross-validated feature selection"
    if missing_rate >= MISSING_THRESHOLD:
        decision = "Remove"
        reason = f"Training-applicant missing rate is at least {MISSING_THRESHOLD:.0%}"
    elif unique_non_missing <= 1:
        decision = "Remove"
        reason = "Constant in the training set where values are available"
    decision_rows.append({
        "feature": feature, "missing_count": int(series.isna().sum()),
        "missing_rate": missing_rate, "unique_non_missing": int(unique_non_missing),
        "pearson_target_correlation": correlation,
        "absolute_correlation": abs(correlation) if pd.notna(correlation) else np.nan,
        "decision": decision, "reason": reason,
    })
feature_decisions = pd.DataFrame(decision_rows).sort_values(
    ["decision", "absolute_correlation"], ascending=[True, False]
).reset_index(drop=True)
removed_features = feature_decisions.loc[feature_decisions["decision"] == "Remove", "feature"].tolist()
bureau_features = bureau_features.drop(columns=removed_features)
print("Features removed:", removed_features)
print("Features retained:", bureau_features.shape[1] - 1)
feature_decisions.round(5)

Features removed: []
Features retained: 50


,feature,missing_count,missing_rate,unique_non_missing,pearson_target_correlation,absolute_correlation,decision,reason
0,BUREAU_DEBT_ABOVE_CREDIT_COUNT,35200,0.14308,8,0.09363,0.09363,Keep,Retain for global cross-validated feature sele...
1,BUREAU_RECENT_CREDIT_COUNT,35200,0.14308,31,0.09135,0.09135,Keep,Retain for global cross-validated feature sele...
2,BUREAU_TOTAL_DEBT_CREDIT_RATIO,36071,0.14663,147978,0.09110,0.09110,Keep,Retain for global cross-validated feature sele...
3,BUREAU_DAYS_CREDIT_MEAN,35200,0.14308,57673,0.09016,0.09016,Keep,Retain for global cross-validated feature sele...
4,BUREAU_CREDIT_AGE_YEARS_MEAN,35200,0.14308,68519,-0.09016,0.09016,Keep,Retain for global cross-validated feature sele...
5,BUREAU_CLOSED_RATE,35200,0.14308,285,-0.07922,0.07922,Keep,Retain for global cross-validated feature sele...
6,BUREAU_ACTIVE_RATE,35200,0.14308,279,0.07744,0.07744,Keep,Retain for global cross-validated feature sele...
7,BUREAU_DAYS_CREDIT_MIN,35200,0.14308,2922,0.07580,0.07580,Keep,Retain for global cross-validated feature sele...
8,BUREAU_ACTIVE_COUNT,35200,0.14308,23,0.06594,0.06594,Keep,Retain for global cross-validated feature sele...
9,BUREAU_DAYS_CREDIT_MAX,35200,0.14308,2922,0.04986,0.04986,Keep,Retain for global cross-validated feature sele...


No bureau features were removed here, all 50 stayed within the missing-value and constant-value limits.


## Validate the applicant-level feature table


In [28]:
numeric_columns = bureau_features.select_dtypes(include="number").columns
infinite_count = sum(int(np.isinf(bureau_features[c].dropna()).sum()) for c in numeric_columns)
retained_decisions = feature_decisions.loc[feature_decisions["decision"] == "Keep"]
validation_checks = pd.DataFrame([
    {"check": "One row per applicant", "passed": bureau_features["SK_ID_CURR"].is_unique},
    {"check": "Only project applicants included", "passed": set(bureau_features["SK_ID_CURR"]).issubset(training_id_set.union(test_id_set))},
    {"check": "No TARGET in feature output", "passed": "TARGET" not in bureau_features.columns},
    {"check": "No bureau account ID in output", "passed": "SK_ID_BUREAU" not in bureau_features.columns},
    {"check": "No infinite numerical values", "passed": infinite_count == 0},
    {"check": "No retained feature reaches 50 percent training missingness", "passed": not retained_decisions["missing_rate"].ge(MISSING_THRESHOLD).any()},
    {"check": "Final test excluded from feature decisions", "passed": not training_base["SK_ID_CURR"].isin(test_id_set).any()},
    {"check": "Account counts are positive", "passed": bureau_features["BUREAU_ACCOUNT_COUNT"].gt(0).all()},
])
assert validation_checks["passed"].all(), "At least one bureau feature-engineering check failed."
validation_checks

,check,passed
0,One row per applicant,True
1,Only project applicants included,True
2,No TARGET in feature output,True
3,No bureau account ID in output,True
4,No infinite numerical values,True
5,No retained feature reaches 50 percent trainin...,True
6,Final test excluded from feature decisions,True
7,Account counts are positive,True


All checks passed.


## Save features and audit reports


In [31]:
bureau_features.to_pickle(output_path)
feature_decisions.to_csv(audit_folder / "bureau_engineered_feature_decisions.csv", index=False)
validation_checks.to_csv(audit_folder / "bureau_feature_engineering_validation.csv", index=False)
print("Bureau feature table saved:", output_path)
print("Output rows:", len(bureau_features))
print("Output columns:", bureau_features.shape[1])
print("Applicants represented:", bureau_features["SK_ID_CURR"].nunique())
print("Remaining numerical missing values:", int(bureau_features.select_dtypes(include="number").isna().sum().sum()))

Bureau feature table saved: /Users/taranveersingh/A-MRP/data/features/bureau_features.pkl
Output rows: 263491
Output columns: 51
Applicants represented: 263491
Remaining numerical missing values: 111165


## Main feature engineering results

This notebook combined the cleaned bureau and bureau-balance data into 50 applicant-level features, covering credit counts, active/closed/overdue status, debt and credit amounts, and credit-type proportions.

All checks passed, and none of the new features needed to be removed. The feature table has 51 columns for the 263,491 applicants who have bureau history. The next step is to build features from the previous-application data.
